# 🏦 Banking Data Platform — Data Profiling & Quality Analysis

This notebook documents the data profiling process — one of the key governance activities
in the pipeline. It visualizes:

- Null rates per column
- Distribution of transaction amounts
- Data quality check results
- Account dormancy analysis
- Credit risk portfolio overview

**Relevant to TD JD:** *'Maintain foundational knowledge of upstream data, including knowledge provided through data profiling, data quality reporting'*

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

TD_GREEN = '#2E7D32'
print('Libraries loaded ✓')

## 1. Load Raw Data (Upstream Extraction)

In [ ]:
RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')

# Try loading staged parquet; fall back to running the extractor
try:
    txn = pd.read_parquet(PROCESSED / 'transactions_staged.parquet')
    acct = pd.read_parquet(PROCESSED / 'accounts_staged.parquet')
    cust = pd.read_parquet(PROCESSED / 'customers_staged.parquet')
    credit = pd.read_parquet(PROCESSED / 'credit_staged.parquet')
    print('Loaded staged parquet files')
except FileNotFoundError:
    print('Staged files not found — running extractor first...')
    from etl.extractors.generate_banking_data import run_extraction
    from etl.transformers.transform import *
    datasets = run_extraction()
    cust   = transform_customers(datasets['customers'])
    acct   = transform_accounts(datasets['accounts'])
    txn    = transform_transactions(datasets['transactions'])
    credit = transform_credit(datasets['credit'])
    print('Extraction and transform complete')

print(f'\nDataset shapes:')
for name, df in [('Transactions', txn), ('Accounts', acct), ('Customers', cust), ('Credit', credit)]:
    print(f'  {name:<15}: {df.shape[0]:>8,} rows × {df.shape[1]:>3} cols')

## 2. Null Rate Profiling (Completeness Dimension)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Null Rate by Column — All Datasets', fontsize=14, fontweight='bold', y=1.01)

datasets = [('Transactions', txn), ('Accounts', acct), ('Customers', cust), ('Credit', credit)]

for ax, (name, df) in zip(axes.flat, datasets):
    null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
    null_pct = null_pct[null_pct.index.str[0] != '_']  # exclude metadata cols
    colors = ['#C62828' if v > 5 else '#F9A825' if v > 0 else TD_GREEN for v in null_pct]
    null_pct.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Null %')
    ax.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='5% threshold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../docs/null_rate_profiling.png', bbox_inches='tight', dpi=150)
plt.show()
print('✓ Null rate chart saved to docs/')

## 3. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Transaction Amount Distribution (CAD)', fontsize=13, fontweight='bold')

# Full distribution
valid_amounts = txn['amount_cad'].dropna()
valid_amounts.clip(upper=5000).hist(bins=60, ax=axes[0], color=TD_GREEN, edgecolor='white')
axes[0].set_title('Full Distribution (clipped at $5K)')
axes[0].set_xlabel('Amount (CAD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# By transaction type
type_stats = txn.groupby('transaction_type')['amount_cad'].median().sort_values()
type_stats.plot(kind='barh', ax=axes[1], color=TD_GREEN)
axes[1].set_title('Median Amount by Type')
axes[1].set_xlabel('Median Amount (CAD)')

# FINTRAC large transactions
large = txn[txn['is_large_transaction'] == True]['amount_cad']
large.clip(upper=100000).hist(bins=30, ax=axes[2], color='#C62828', edgecolor='white')
axes[2].set_title(f'FINTRAC Reportable (n={len(large):,})')
axes[2].set_xlabel('Amount (CAD)')

plt.tight_layout()
plt.savefig('../docs/transaction_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Data Quality Results Visualization

In [ ]:
from etl.transformers.data_quality import TransactionValidator, AccountValidator, generate_report

# Run validators
raw_txn  = pd.read_csv('../data/raw/transactions_raw.csv') if Path('../data/raw/transactions_raw.csv').exists() else txn
raw_acct = pd.read_csv('../data/raw/accounts_raw.csv')     if Path('../data/raw/accounts_raw.csv').exists()     else acct

acct_ids = set(raw_acct['account_id']) if 'account_id' in raw_acct.columns else set(acct['account_id'])
cust_ids = set(cust['customer_id'])

txn_validator  = TransactionValidator(acct_ids)
acct_validator = AccountValidator(cust_ids)

_, txn_checks  = txn_validator.validate(raw_txn)
_, acct_checks = acct_validator.validate(raw_acct)

def checks_to_df(checks, dataset_name):
    return pd.DataFrame([{
        'Dataset': dataset_name,
        'Check': c.check_name,
        'Dimension': c.dimension,
        'Passed': c.passed,
        'Failed Count': c.failed_count,
        'Failed %': c.failed_pct,
        'Action': c.action,
    } for c in checks])

dq_df = pd.concat([checks_to_df(txn_checks, 'Transactions'), checks_to_df(acct_checks, 'Accounts')])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Data Quality Check Results', fontsize=13, fontweight='bold')

# Pass/Fail counts by dimension
dim_pass = dq_df.groupby(['Dimension', 'Passed']).size().unstack(fill_value=0)
dim_pass.plot(kind='bar', ax=axes[0], color=[TD_GREEN, '#C62828'], edgecolor='white')
axes[0].set_title('Checks by Dimension')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(['Pass', 'Fail'])

# Failed % heatmap
failed_only = dq_df[dq_df['Failed Count'] > 0][['Check','Failed %','Action']]
if len(failed_only) > 0:
    colors = ['#C62828' if a == 'QUARANTINE' else '#F9A825' for a in failed_only['Action']]
    axes[1].barh(failed_only['Check'], failed_only['Failed %'], color=colors)
    axes[1].set_title('Failed Checks (% of rows affected)')
    axes[1].set_xlabel('Failed %')
else:
    axes[1].text(0.5, 0.5, '✓ All checks passed!', ha='center', va='center',
                 fontsize=14, color=TD_GREEN, transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig('../docs/dq_results.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Total checks run: {len(dq_df)} | Passed: {dq_df["Passed"].sum()} | Failed: {(~dq_df["Passed"]).sum()}')

## 5. Account Dormancy Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Account Dormancy Analysis', fontsize=13, fontweight='bold')

# Dormancy rate by account type
if 'is_dormant' in acct.columns and 'account_type' in acct.columns:
    dormancy_rate = acct.groupby('account_type')['is_dormant'].mean() * 100
    dormancy_rate.sort_values().plot(kind='barh', ax=axes[0], color='#F9A825')
    axes[0].axvline(x=dormancy_rate.mean(), color='red', linestyle='--',
                    label=f'Avg: {dormancy_rate.mean():.1f}%')
    axes[0].set_title('Dormancy Rate by Account Type')
    axes[0].set_xlabel('Dormancy Rate (%)')
    axes[0].legend()

# Days since activity distribution
if 'days_since_activity' in acct.columns:
    acct['days_since_activity'].clip(upper=1500).hist(
        bins=50, ax=axes[1], color=TD_GREEN, edgecolor='white'
    )
    axes[1].axvline(x=730, color='red', linestyle='--', label='730-day threshold')
    axes[1].set_title('Days Since Last Activity')
    axes[1].set_xlabel('Days')
    axes[1].legend()

plt.tight_layout()
plt.savefig('../docs/dormancy_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Summary — Key Findings for Stakeholders

In [ ]:
print('=' * 60)
print('  DATA PROFILING SUMMARY — Key Findings')
print('=' * 60)

print(f'\n  TRANSACTIONS ({len(txn):,} records)')
print(f'  Total volume     : ${txn["amount_cad"].sum():>15,.2f} CAD')
print(f'  Avg transaction  : ${txn["amount_cad"].mean():>15,.2f} CAD')
print(f'  FINTRAC reportable: {txn["is_large_transaction"].sum():>14,} transactions')
print(f'  Failure rate     : {(txn["status"]=="Failed").mean()*100:>14.1f}%')

if 'is_dormant' in acct.columns:
    print(f'\n  ACCOUNTS ({len(acct):,} records)')
    print(f'  Dormancy rate    : {acct["is_dormant"].mean()*100:>14.1f}%')
    print(f'  Overdraft rate   : {acct["is_overdraft"].mean()*100:>14.1f}%')
    print(f'  Total deposits   : ${acct[acct["current_balance"]>0]["current_balance"].sum():>15,.0f} CAD')

if 'is_delinquent' in credit.columns:
    print(f'\n  CREDIT PORTFOLIO ({len(credit):,} loans)')
    print(f'  Delinquency rate : {credit["is_delinquent"].mean()*100:>14.1f}%')
    print(f'  Total exposure   : ${credit["outstanding_balance"].sum():>15,.0f} CAD')
    print(f'  Avg LTV          : {credit["ltv_ratio"].mean():>15.2%}')

print('\n' + '=' * 60)